In [ ]:

import os
import joblib
import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt

# All proteins

In [ ]:
save_dir = "./SHAP_Data/All"
fold = 1
# 学習済みモデル
model = joblib.load(
    os.path.join(save_dir, f"rf_model_fold_{fold}.joblib")
)

# テストデータ
test_data = pd.read_csv(
    os.path.join(save_dir, f"test_data_fold_{fold}.csv"),
    index_col="SampleID"
)


In [ ]:
print(model.classes_)

In [ ]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

# ==========================================
# 1. 保存先フォルダ
# ==========================================
save_dir = "./SHAP_Data/All"

# 特徴量名を読み込む
with open(
    os.path.join(save_dir, "feature_names.json"),
    "r",
    encoding="utf-8"
) as f:
    feature_names = json.load(f)

# ==========================================
# 2. FoldごとのSHAP値を計算・保存
# ==========================================
for fold in range(1, 6):

    print(f"\n========== Fold {fold} ==========")

    # 学習済みモデルを読み込む
    model = joblib.load(
        os.path.join(save_dir, f"rf_model_fold_{fold}.joblib")
    )

    # テストデータを読み込む
    test_data = pd.read_csv(
        os.path.join(save_dir, f"test_data_fold_{fold}.csv"),
        index_col="SampleID"
    )

    # 正解ラベルを除き、学習時と同じ特徴量・列順にする
    X_test = test_data.drop(columns=["__true_label__"])
    X_test = X_test.loc[:, feature_names]

    # SHAP値を計算
    explainer = shap.TreeExplainer(model)
    shap_exp = explainer(X_test)

    values = shap_exp.values
    base_values = shap_exp.base_values

    # クラス1（陽性）のSHAP値を取得
    if values.ndim == 3:
        shap_class1 = values[:, :, 1]

        if np.ndim(base_values) == 2:
            base_class1 = base_values[:, 1]
        else:
            base_class1 = base_values

    elif values.ndim == 2:
        shap_class1 = values

        if np.ndim(base_values) == 2:
            base_class1 = base_values[:, 1]
        else:
            base_class1 = base_values

    else:
        raise ValueError(f"想定外のSHAP shape: {values.shape}")

    print("SHAP shape:", shap_class1.shape)

    # SHAP値・base valueを保存
    np.save(
        os.path.join(save_dir, f"shap_values_fold_{fold}.npy"),
        shap_class1
    )

    np.save(
        os.path.join(save_dir, f"shap_base_values_fold_{fold}.npy"),
        base_class1
    )

    # サンプルID・タンパク質名付きCSV
    pd.DataFrame(
        shap_class1,
        index=X_test.index,
        columns=feature_names
    ).to_csv(
        os.path.join(save_dir, f"shap_values_fold_{fold}.csv"),
        index_label="SampleID"
    )

    # beeswarm plotを保存
    shap_class1_exp = shap.Explanation(
        values=shap_class1,
        base_values=base_class1,
        data=X_test.to_numpy(),
        feature_names=feature_names
    )

    shap.plots.beeswarm(
        shap_class1_exp,
        max_display=20,
        show=False
    )

    plt.title(f"SHAP Beeswarm - All Proteins - Fold {fold}")
    plt.tight_layout()

    plt.savefig(
        os.path.join(save_dir, f"shap_beeswarm_fold_{fold}.png"),
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()
    plt.close()

# ==========================================
# 3. 5 Foldの平均絶対SHAP値を計算
# ==========================================
fold_mean_abs_shap = []

for fold in range(1, 6):

    shap_values = np.load(
        os.path.join(save_dir, f"shap_values_fold_{fold}.npy")
    )

    # サンプルごとの絶対SHAP値を平均
    mean_abs_shap = np.abs(shap_values).mean(axis=0)
    fold_mean_abs_shap.append(mean_abs_shap)

fold_mean_abs_shap = np.asarray(fold_mean_abs_shap)

print("\nFold × Protein:", fold_mean_abs_shap.shape)

# ==========================================
# 4. 平均・標準偏差・順位・安定性を計算
# ==========================================
importance_mean = fold_mean_abs_shap.mean(axis=0)
importance_sd = fold_mean_abs_shap.std(axis=0, ddof=1)

fold_ranks = np.array([
    pd.Series(values, index=feature_names)
      .rank(ascending=False, method="min")
      .to_numpy()
    for values in fold_mean_abs_shap
])

mean_rank = fold_ranks.mean(axis=0)
rank_sd = fold_ranks.std(axis=0, ddof=1)
top20_count = (fold_ranks <= 20).sum(axis=0)

shap_summary = pd.DataFrame({
    "Protein": feature_names,
    "MeanAbsSHAP_Mean": importance_mean,
    "MeanAbsSHAP_SD": importance_sd,
    "MeanRank": mean_rank,
    "Rank_SD": rank_sd,
    "Top20_Fold_Count": top20_count
})

shap_summary = shap_summary.sort_values(
    "MeanAbsSHAP_Mean",
    ascending=False
).reset_index(drop=True)

shap_summary.insert(
    0,
    "Overall_Rank",
    np.arange(1, len(shap_summary) + 1)
)

# 結果を保存
shap_summary.to_csv(
    os.path.join(save_dir, "SHAP_importance_5fold.csv"),
    index=False
)

display(shap_summary.head(20).round(4))

# ==========================================
# 5. 上位20タンパク質のグラフ
# ==========================================
top20 = shap_summary.head(20).sort_values(
    "MeanAbsSHAP_Mean",
    ascending=True
)

plt.figure(figsize=(3, 4))

plt.barh(
    top20["Protein"],
    top20["MeanAbsSHAP_Mean"],
    xerr=top20["MeanAbsSHAP_SD"],
    capsize=3
)

plt.xlabel("Mean |SHAP value| across 5 folds")
plt.ylabel("Protein")
plt.title("Top 20 Proteins by SHAP Importance - All Proteins")

plt.tight_layout()

plt.savefig(
    os.path.join(save_dir, "SHAP_top20_importance.png"),
    dpi=600,
    bbox_inches="tight"
)

plt.show()
plt.close()

# ==========================================
# 6. Fold間で安定して重要なタンパク質を抽出
# ==========================================
stable_proteins = shap_summary[
    (shap_summary["Top20_Fold_Count"] >= 4) &
    (shap_summary["Rank_SD"] <= 10)
].sort_values(
    "MeanAbsSHAP_Mean",
    ascending=False
)

stable_proteins.to_csv(
    os.path.join(save_dir, "SHAP_stable_proteins.csv"),
    index=False
)

print("\n安定性の高い候補タンパク質数:", len(stable_proteins))
display(stable_proteins.head(30).round(4))

print("\n全処理が完了しました。保存先:", os.path.abspath(save_dir))

In [ ]:
# SHAP値を計算
explainer = shap.TreeExplainer(model)
shap_values = explainer(X_test)

print("SHAP shape:", shap_values.values.shape)

# クラス1（陽性）のSHAP値を抽出
shap_values_class1 = shap_values.values[:, :, 1]

# 保存
np.save(
    os.path.join(save_dir, f"shap_values_fold_{fold}.npy"),
    shap_values_class1
)

np.save(
    os.path.join(save_dir, f"shap_base_values_fold_{fold}.npy"),
    shap_values.base_values[:, 1]
)

# タンパク質名・サンプルID付きCSV
shap_df = pd.DataFrame(
    shap_values_class1,
    index=X_test.index,
    columns=X_test.columns
)

shap_df.to_csv(
    os.path.join(save_dir, f"shap_values_fold_{fold}.csv"),
    index_label="SampleID"
)

# クラス1のSHAP値を使って可視化
shap_class1 = shap.Explanation(
    values=shap_values_class1,
    base_values=shap_values.base_values[:, 1],
    data=X_test.to_numpy(),
    feature_names=X_test.columns.tolist()
)

shap.plots.beeswarm(shap_class1, max_display=20)

plt.tight_layout()
plt.savefig(
    os.path.join(save_dir, f"shap_beeswarm_fold_{fold}.png"),
    dpi=300,
    bbox_inches="tight"
)
plt.show()

In [ ]:
save_dir = "./SHAP_Data/DEP"
fold = 1
# 学習済みモデル
model = joblib.load(
    os.path.join(save_dir, f"rf_model_fold_{fold}.joblib")
)

# テストデータ
test_data = pd.read_csv(
    os.path.join(save_dir, f"test_data_fold_{fold}.csv"),
    index_col="SampleID"
)


In [ ]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

# ==========================================
# 1. 保存先フォルダ
# ==========================================
save_dir = "./SHAP_Data/DEP"

# 特徴量名を読み込む
with open(
    os.path.join(save_dir, "feature_names.json"),
    "r",
    encoding="utf-8"
) as f:
    feature_names = json.load(f)

# ==========================================
# 2. FoldごとのSHAP値を計算・保存
# ==========================================
for fold in range(1, 6):

    print(f"\n========== Fold {fold} ==========")

    # 学習済みモデルを読み込む
    model = joblib.load(
        os.path.join(save_dir, f"rf_model_fold_{fold}.joblib")
    )

    # テストデータを読み込む
    test_data = pd.read_csv(
        os.path.join(save_dir, f"test_data_fold_{fold}.csv"),
        index_col="SampleID"
    )

    # 正解ラベルを除き、学習時と同じ特徴量・列順にする
    X_test = test_data.drop(columns=["__true_label__"])
    X_test = X_test.loc[:, feature_names]

    # SHAP値を計算
    explainer = shap.TreeExplainer(model)
    shap_exp = explainer(X_test)

    values = shap_exp.values
    base_values = shap_exp.base_values

    # クラス1（陽性）のSHAP値を取得
    if values.ndim == 3:
        shap_class1 = values[:, :, 1]

        if np.ndim(base_values) == 2:
            base_class1 = base_values[:, 1]
        else:
            base_class1 = base_values

    elif values.ndim == 2:
        shap_class1 = values

        if np.ndim(base_values) == 2:
            base_class1 = base_values[:, 1]
        else:
            base_class1 = base_values

    else:
        raise ValueError(f"想定外のSHAP shape: {values.shape}")

    print("SHAP shape:", shap_class1.shape)

    # SHAP値・base valueを保存
    np.save(
        os.path.join(save_dir, f"shap_values_fold_{fold}.npy"),
        shap_class1
    )

    np.save(
        os.path.join(save_dir, f"shap_base_values_fold_{fold}.npy"),
        base_class1
    )

    # サンプルID・タンパク質名付きCSV
    pd.DataFrame(
        shap_class1,
        index=X_test.index,
        columns=feature_names
    ).to_csv(
        os.path.join(save_dir, f"shap_values_fold_{fold}.csv"),
        index_label="SampleID"
    )

    # beeswarm plotを保存
    shap_class1_exp = shap.Explanation(
        values=shap_class1,
        base_values=base_class1,
        data=X_test.to_numpy(),
        feature_names=feature_names
    )

    shap.plots.beeswarm(
        shap_class1_exp,
        max_display=20,
        show=False
    )

    plt.title(f"SHAP Beeswarm - All Proteins - Fold {fold}")
    plt.tight_layout()

    plt.savefig(
        os.path.join(save_dir, f"shap_beeswarm_fold_{fold}.png"),
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()
    plt.close()

# ==========================================
# 3. 5 Foldの平均絶対SHAP値を計算
# ==========================================
fold_mean_abs_shap = []

for fold in range(1, 6):

    shap_values = np.load(
        os.path.join(save_dir, f"shap_values_fold_{fold}.npy")
    )

    # サンプルごとの絶対SHAP値を平均
    mean_abs_shap = np.abs(shap_values).mean(axis=0)
    fold_mean_abs_shap.append(mean_abs_shap)

fold_mean_abs_shap = np.asarray(fold_mean_abs_shap)

print("\nFold × Protein:", fold_mean_abs_shap.shape)

# ==========================================
# 4. 平均・標準偏差・順位・安定性を計算
# ==========================================
importance_mean = fold_mean_abs_shap.mean(axis=0)
importance_sd = fold_mean_abs_shap.std(axis=0, ddof=1)

fold_ranks = np.array([
    pd.Series(values, index=feature_names)
      .rank(ascending=False, method="min")
      .to_numpy()
    for values in fold_mean_abs_shap
])

mean_rank = fold_ranks.mean(axis=0)
rank_sd = fold_ranks.std(axis=0, ddof=1)
top20_count = (fold_ranks <= 20).sum(axis=0)

shap_summary = pd.DataFrame({
    "Protein": feature_names,
    "MeanAbsSHAP_Mean": importance_mean,
    "MeanAbsSHAP_SD": importance_sd,
    "MeanRank": mean_rank,
    "Rank_SD": rank_sd,
    "Top20_Fold_Count": top20_count
})

shap_summary = shap_summary.sort_values(
    "MeanAbsSHAP_Mean",
    ascending=False
).reset_index(drop=True)

shap_summary.insert(
    0,
    "Overall_Rank",
    np.arange(1, len(shap_summary) + 1)
)

# 結果を保存
shap_summary.to_csv(
    os.path.join(save_dir, "SHAP_importance_5fold.csv"),
    index=False
)

display(shap_summary.head(20).round(4))

# ==========================================
# 5. 上位20タンパク質のグラフ
# ==========================================
top20 = shap_summary.head(20).sort_values(
    "MeanAbsSHAP_Mean",
    ascending=True
)

plt.figure(figsize=(3, 4))

plt.barh(
    top20["Protein"],
    top20["MeanAbsSHAP_Mean"],
    xerr=top20["MeanAbsSHAP_SD"],
    capsize=3
)

plt.xlabel("Mean |SHAP value| across 5 folds")
plt.ylabel("Protein")
plt.title("Top 20 Proteins by SHAP Importance - All Proteins")

plt.tight_layout()

plt.savefig(
    os.path.join(save_dir, "SHAP_top20_importance.png"),
    dpi=600,
    bbox_inches="tight"
)

plt.show()
plt.close()

# ==========================================
# 6. Fold間で安定して重要なタンパク質を抽出
# ==========================================
stable_proteins = shap_summary[
    (shap_summary["Top20_Fold_Count"] >= 4) &
    (shap_summary["Rank_SD"] <= 10)
].sort_values(
    "MeanAbsSHAP_Mean",
    ascending=False
)

stable_proteins.to_csv(
    os.path.join(save_dir, "SHAP_stable_proteins.csv"),
    index=False
)

print("\n安定性の高い候補タンパク質数:", len(stable_proteins))
display(stable_proteins.head(30).round(4))

print("\n全処理が完了しました。保存先:", os.path.abspath(save_dir))

# DEP

In [ ]:
save_dir = "./SHAP_Data/DEP"
fold = 1
# 学習済みモデル
model = joblib.load(
    os.path.join(save_dir, f"rf_model_fold_{fold}.joblib")
)

# テストデータ
test_data = pd.read_csv(
    os.path.join(save_dir, f"test_data_fold_{fold}.csv"),
    index_col="SampleID"
)


In [ ]:
print(model.classes_)

In [ ]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

# ==========================================
# 1. 保存先フォルダ
# ==========================================
save_dir = "./SHAP_Data/DEP"

# 特徴量名を読み込む
with open(
    os.path.join(save_dir, "feature_names.json"),
    "r",
    encoding="utf-8"
) as f:
    feature_names = json.load(f)

# ==========================================
# 2. FoldごとのSHAP値を計算・保存
# ==========================================
for fold in range(1, 6):

    print(f"\n========== Fold {fold} ==========")

    # 学習済みモデルを読み込む
    model = joblib.load(
        os.path.join(save_dir, f"rf_model_fold_{fold}.joblib")
    )

    # テストデータを読み込む
    test_data = pd.read_csv(
        os.path.join(save_dir, f"test_data_fold_{fold}.csv"),
        index_col="SampleID"
    )

    # 正解ラベルを除き、学習時と同じ特徴量・列順にする
    X_test = test_data.drop(columns=["__true_label__"])
    X_test = X_test.loc[:, feature_names]

    # SHAP値を計算
    explainer = shap.TreeExplainer(model)
    shap_exp = explainer(X_test)

    values = shap_exp.values
    base_values = shap_exp.base_values

    # クラス1（陽性）のSHAP値を取得
    if values.ndim == 3:
        shap_class1 = values[:, :, 1]

        if np.ndim(base_values) == 2:
            base_class1 = base_values[:, 1]
        else:
            base_class1 = base_values

    elif values.ndim == 2:
        shap_class1 = values

        if np.ndim(base_values) == 2:
            base_class1 = base_values[:, 1]
        else:
            base_class1 = base_values

    else:
        raise ValueError(f"想定外のSHAP shape: {values.shape}")

    print("SHAP shape:", shap_class1.shape)

    # SHAP値・base valueを保存
    np.save(
        os.path.join(save_dir, f"shap_values_fold_{fold}.npy"),
        shap_class1
    )

    np.save(
        os.path.join(save_dir, f"shap_base_values_fold_{fold}.npy"),
        base_class1
    )

    # サンプルID・タンパク質名付きCSV
    pd.DataFrame(
        shap_class1,
        index=X_test.index,
        columns=feature_names
    ).to_csv(
        os.path.join(save_dir, f"shap_values_fold_{fold}.csv"),
        index_label="SampleID"
    )

    # beeswarm plotを保存
    shap_class1_exp = shap.Explanation(
        values=shap_class1,
        base_values=base_class1,
        data=X_test.to_numpy(),
        feature_names=feature_names
    )

    shap.plots.beeswarm(
        shap_class1_exp,
        max_display=20,
        show=False
    )

    plt.title(f"SHAP Beeswarm - All Proteins - Fold {fold}")
    plt.tight_layout()

    plt.savefig(
        os.path.join(save_dir, f"shap_beeswarm_fold_{fold}.png"),
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()
    plt.close()

# ==========================================
# 3. 5 Foldの平均絶対SHAP値を計算
# ==========================================
fold_mean_abs_shap = []

for fold in range(1, 6):

    shap_values = np.load(
        os.path.join(save_dir, f"shap_values_fold_{fold}.npy")
    )

    # サンプルごとの絶対SHAP値を平均
    mean_abs_shap = np.abs(shap_values).mean(axis=0)
    fold_mean_abs_shap.append(mean_abs_shap)

fold_mean_abs_shap = np.asarray(fold_mean_abs_shap)

print("\nFold × Protein:", fold_mean_abs_shap.shape)

# ==========================================
# 4. 平均・標準偏差・順位・安定性を計算
# ==========================================
importance_mean = fold_mean_abs_shap.mean(axis=0)
importance_sd = fold_mean_abs_shap.std(axis=0, ddof=1)

fold_ranks = np.array([
    pd.Series(values, index=feature_names)
      .rank(ascending=False, method="min")
      .to_numpy()
    for values in fold_mean_abs_shap
])

mean_rank = fold_ranks.mean(axis=0)
rank_sd = fold_ranks.std(axis=0, ddof=1)
top20_count = (fold_ranks <= 20).sum(axis=0)

shap_summary = pd.DataFrame({
    "Protein": feature_names,
    "MeanAbsSHAP_Mean": importance_mean,
    "MeanAbsSHAP_SD": importance_sd,
    "MeanRank": mean_rank,
    "Rank_SD": rank_sd,
    "Top20_Fold_Count": top20_count
})

shap_summary = shap_summary.sort_values(
    "MeanAbsSHAP_Mean",
    ascending=False
).reset_index(drop=True)

shap_summary.insert(
    0,
    "Overall_Rank",
    np.arange(1, len(shap_summary) + 1)
)

# 結果を保存
shap_summary.to_csv(
    os.path.join(save_dir, "SHAP_importance_5fold.csv"),
    index=False
)

display(shap_summary.head(20).round(4))

# ==========================================
# 5. 上位20タンパク質のグラフ
# ==========================================
top20 = shap_summary.head(20).sort_values(
    "MeanAbsSHAP_Mean",
    ascending=True
)

plt.figure(figsize=(3, 4))

plt.barh(
    top20["Protein"],
    top20["MeanAbsSHAP_Mean"],
    xerr=top20["MeanAbsSHAP_SD"],
    capsize=3
)

plt.xlabel("Mean |SHAP value| across 5 folds")
plt.ylabel("Protein")
plt.title("Top 20 Proteins by SHAP Importance - All Proteins")

plt.tight_layout()

plt.savefig(
    os.path.join(save_dir, "SHAP_top20_importance.png"),
    dpi=600,
    bbox_inches="tight"
)

plt.show()
plt.close()

# ==========================================
# 6. Fold間で安定して重要なタンパク質を抽出
# ==========================================
stable_proteins = shap_summary[
    (shap_summary["Top20_Fold_Count"] >= 4) &
    (shap_summary["Rank_SD"] <= 10)
].sort_values(
    "MeanAbsSHAP_Mean",
    ascending=False
)

stable_proteins.to_csv(
    os.path.join(save_dir, "SHAP_stable_proteins.csv"),
    index=False
)

print("\n安定性の高い候補タンパク質数:", len(stable_proteins))
display(stable_proteins.head(30).round(4))

print("\n全処理が完了しました。保存先:", os.path.abspath(save_dir))

In [ ]:
# SHAP値を計算
explainer = shap.TreeExplainer(model)
shap_values = explainer(X_test)

print("SHAP shape:", shap_values.values.shape)

# クラス1（陽性）のSHAP値を抽出
shap_values_class1 = shap_values.values[:, :, 1]

# 保存
np.save(
    os.path.join(save_dir, f"shap_values_fold_{fold}.npy"),
    shap_values_class1
)

np.save(
    os.path.join(save_dir, f"shap_base_values_fold_{fold}.npy"),
    shap_values.base_values[:, 1]
)

# タンパク質名・サンプルID付きCSV
shap_df = pd.DataFrame(
    shap_values_class1,
    index=X_test.index,
    columns=X_test.columns
)

shap_df.to_csv(
    os.path.join(save_dir, f"shap_values_fold_{fold}.csv"),
    index_label="SampleID"
)

# クラス1のSHAP値を使って可視化
shap_class1 = shap.Explanation(
    values=shap_values_class1,
    base_values=shap_values.base_values[:, 1],
    data=X_test.to_numpy(),
    feature_names=X_test.columns.tolist()
)

shap.plots.beeswarm(shap_class1, max_display=20)

plt.tight_layout()
plt.savefig(
    os.path.join(save_dir, f"shap_beeswarm_fold_{fold}.png"),
    dpi=300,
    bbox_inches="tight"
)
plt.show()

# 二つを統合

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# ==========================================
# 1. データ読み込み
# ==========================================
all_dir = "./SHAP_Data/All"
dep_dir = "./SHAP_Data/DEP"

all_df = pd.read_csv(
    os.path.join(all_dir, "SHAP_importance_5fold.csv")
)
dep_df = pd.read_csv(
    os.path.join(dep_dir, "SHAP_importance_5fold.csv")
)
all_df["Protein"] = all_df["Protein"].replace(
    "Q14213_Q8NEV9*", "EBI3_IL27"
)

dep_df["Protein"] = dep_df["Protein"].replace(
    "Q14213_Q8NEV9*", "EBI3_IL27"
)
# それぞれの上位20タンパク質を抽出
all_top20 = all_df.nlargest(10, "MeanAbsSHAP_Mean").copy()
dep_top20 = dep_df.nlargest(10, "MeanAbsSHAP_Mean").copy()

# ==========================================
# 2. 共通・固有タンパク質の分類
# ==========================================
all_proteins = set(all_top20["Protein"])
dep_proteins = set(dep_top20["Protein"])

common = all_proteins & dep_proteins
all_only = all_proteins - dep_proteins
dep_only = dep_proteins - all_proteins

print("共通タンパク質数:", len(common))
print("Allのみ:", len(all_only))
print("DEPのみ:", len(dep_only))

# ==========================================
# 3. データを結合（平均は取らない）
# ==========================================
all_top20["Dataset"] = "All"
dep_top20["Dataset"] = "DEP"

combined = pd.concat(
    [all_top20, dep_top20],
    ignore_index=True
)

# Allは赤、DEPは青に固定
color_map = {
    "All": "#FF4B00",
    "DEP": "#005AFF"
}
combined["Color"] = combined["Dataset"].map(color_map)

# ==========================================
# 4. タンパク質ごとにAll・DEPの棒を表示
# ==========================================
protein_order = (
    combined.groupby("Protein")["MeanAbsSHAP_Mean"]
    .max()
    .sort_values()
    .index.tolist()
)

fig, ax = plt.subplots(
    figsize=(6, 8)
)
plt.rcParams.update({
    "font.size": 14,          # 基本の文字サイズ
    "axes.titlesize": 18,     # グラフタイトル
    "axes.labelsize": 16,     # 軸ラベル
    "xtick.labelsize": 14,    # 横軸の目盛り
    "ytick.labelsize": 24,    # 縦軸（タンパク質名）
    "legend.fontsize": 14     # 凡例
})
bar_height = 0.35
y_positions = {
    protein: i for i, protein in enumerate(protein_order)
}

for _, row in combined.iterrows():
    protein = row["Protein"]
    dataset = row["Dataset"]
    value = row["MeanAbsSHAP_Mean"]
    sd = row["MeanAbsSHAP_SD"]
    # Allは上側、DEPは下側に表示
    offset = (
        bar_height / 2
        if dataset == "All"
        else -bar_height / 2
    )

    ax.barh(
        y_positions[protein] + offset,
        value,
        xerr=sd,
        height=bar_height,
        error_kw={"elinewidth": 1, "capthick": 1},
        color=color_map[dataset],
        edgecolor="black",
        linewidth=0.4
    )

    # ax.text(
    #     value,
    #     y_positions[protein] + offset,
    #     f" {dataset}",
    #     va="center",
    #     fontsize=7
    # )

ax.set_yticks(range(len(protein_order)))
ax.set_yticklabels(protein_order)
ax.set_xlabel("Mean |SHAP value| across 5 folds")
ax.set_ylabel("Protein")
ax.set_title("SHAP Importance: All vs DEP")
ax.set_yticklabels(protein_order, fontsize=20)
# 凡例もAll＝赤、DEP＝青に固定
legend_items = [
    Patch(facecolor="#FF4B00", edgecolor="black", label="All"),
    Patch(facecolor="#005AFF", edgecolor="black", label="DEP")
]

ax.legend(handles=legend_items, loc="lower right")

plt.tight_layout()

save_path = "./SHAP_Data/SHAP_All_vs_DEP_top20.png"
plt.savefig(save_path, dpi=600, bbox_inches="tight",transparent=True)
plt.show()

# All + ca19-9

In [ ]:
save_dir = "./SHAP_Data/Allca19-9"
fold = 1
# 学習済みモデル
model = joblib.load(
    os.path.join(save_dir, f"rf_model_fold_{fold}.joblib")
)

# テストデータ
test_data = pd.read_csv(
    os.path.join(save_dir, f"test_data_fold_{fold}.csv"),
    index_col="SampleID"
)


In [ ]:
print(model.classes_)

In [ ]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

# ==========================================
# 1. 保存先フォルダ
# ==========================================
save_dir = "./SHAP_Data/Allca19-9"

# 特徴量名を読み込む
with open(
    os.path.join(save_dir, "feature_names.json"),
    "r",
    encoding="utf-8"
) as f:
    feature_names = json.load(f)

# ==========================================
# 2. FoldごとのSHAP値を計算・保存
# ==========================================
for fold in range(1, 6):

    print(f"\n========== Fold {fold} ==========")

    # 学習済みモデルを読み込む
    model = joblib.load(
        os.path.join(save_dir, f"rf_model_fold_{fold}.joblib")
    )

    # テストデータを読み込む
    test_data = pd.read_csv(
        os.path.join(save_dir, f"test_data_fold_{fold}.csv"),
        index_col="SampleID"
    )

    # 正解ラベルを除き、学習時と同じ特徴量・列順にする
    X_test = test_data.drop(columns=["__true_label__"])
    X_test = X_test.loc[:, feature_names]

    # SHAP値を計算
    explainer = shap.TreeExplainer(model)
    shap_exp = explainer(X_test)

    values = shap_exp.values
    base_values = shap_exp.base_values

    # クラス1（陽性）のSHAP値を取得
    if values.ndim == 3:
        shap_class1 = values[:, :, 1]

        if np.ndim(base_values) == 2:
            base_class1 = base_values[:, 1]
        else:
            base_class1 = base_values

    elif values.ndim == 2:
        shap_class1 = values

        if np.ndim(base_values) == 2:
            base_class1 = base_values[:, 1]
        else:
            base_class1 = base_values

    else:
        raise ValueError(f"想定外のSHAP shape: {values.shape}")

    print("SHAP shape:", shap_class1.shape)

    # SHAP値・base valueを保存
    np.save(
        os.path.join(save_dir, f"shap_values_fold_{fold}.npy"),
        shap_class1
    )

    np.save(
        os.path.join(save_dir, f"shap_base_values_fold_{fold}.npy"),
        base_class1
    )

    # サンプルID・タンパク質名付きCSV
    pd.DataFrame(
        shap_class1,
        index=X_test.index,
        columns=feature_names
    ).to_csv(
        os.path.join(save_dir, f"shap_values_fold_{fold}.csv"),
        index_label="SampleID"
    )

    # beeswarm plotを保存
    shap_class1_exp = shap.Explanation(
        values=shap_class1,
        base_values=base_class1,
        data=X_test.to_numpy(),
        feature_names=feature_names
    )

    shap.plots.beeswarm(
        shap_class1_exp,
        max_display=20,
        show=False
    )

    plt.title(f"SHAP Beeswarm - All Proteins - Fold {fold}")
    plt.tight_layout()

    plt.savefig(
        os.path.join(save_dir, f"shap_beeswarm_fold_{fold}.png"),
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()
    plt.close()

# ==========================================
# 3. 5 Foldの平均絶対SHAP値を計算
# ==========================================
fold_mean_abs_shap = []

for fold in range(1, 6):

    shap_values = np.load(
        os.path.join(save_dir, f"shap_values_fold_{fold}.npy")
    )

    # サンプルごとの絶対SHAP値を平均
    mean_abs_shap = np.abs(shap_values).mean(axis=0)
    fold_mean_abs_shap.append(mean_abs_shap)

fold_mean_abs_shap = np.asarray(fold_mean_abs_shap)

print("\nFold × Protein:", fold_mean_abs_shap.shape)

# ==========================================
# 4. 平均・標準偏差・順位・安定性を計算
# ==========================================
importance_mean = fold_mean_abs_shap.mean(axis=0)
importance_sd = fold_mean_abs_shap.std(axis=0, ddof=1)

fold_ranks = np.array([
    pd.Series(values, index=feature_names)
      .rank(ascending=False, method="min")
      .to_numpy()
    for values in fold_mean_abs_shap
])

mean_rank = fold_ranks.mean(axis=0)
rank_sd = fold_ranks.std(axis=0, ddof=1)
top20_count = (fold_ranks <= 20).sum(axis=0)

shap_summary = pd.DataFrame({
    "Protein": feature_names,
    "MeanAbsSHAP_Mean": importance_mean,
    "MeanAbsSHAP_SD": importance_sd,
    "MeanRank": mean_rank,
    "Rank_SD": rank_sd,
    "Top20_Fold_Count": top20_count
})

shap_summary = shap_summary.sort_values(
    "MeanAbsSHAP_Mean",
    ascending=False
).reset_index(drop=True)

shap_summary.insert(
    0,
    "Overall_Rank",
    np.arange(1, len(shap_summary) + 1)
)

# 結果を保存
shap_summary.to_csv(
    os.path.join(save_dir, "SHAP_importance_5fold.csv"),
    index=False
)

display(shap_summary.head(20).round(4))

# ==========================================
# 5. 上位20タンパク質のグラフ
# ==========================================
top20 = shap_summary.head(20).sort_values(
    "MeanAbsSHAP_Mean",
    ascending=True
)

plt.figure(figsize=(3, 4))

plt.barh(
    top20["Protein"],
    top20["MeanAbsSHAP_Mean"],
    xerr=top20["MeanAbsSHAP_SD"],
    capsize=3
)

plt.xlabel("Mean |SHAP value| across 5 folds")
plt.ylabel("Protein")
plt.title("Top 20 Proteins by SHAP Importance - All Proteins")

plt.tight_layout()

plt.savefig(
    os.path.join(save_dir, "SHAP_top20_importance.png"),
    dpi=600,
    bbox_inches="tight"
)

plt.show()
plt.close()

# ==========================================
# 6. Fold間で安定して重要なタンパク質を抽出
# ==========================================
stable_proteins = shap_summary[
    (shap_summary["Top20_Fold_Count"] >= 4) &
    (shap_summary["Rank_SD"] <= 10)
].sort_values(
    "MeanAbsSHAP_Mean",
    ascending=False
)

stable_proteins.to_csv(
    os.path.join(save_dir, "SHAP_stable_proteins.csv"),
    index=False
)

print("\n安定性の高い候補タンパク質数:", len(stable_proteins))
display(stable_proteins.head(30).round(4))

print("\n全処理が完了しました。保存先:", os.path.abspath(save_dir))